In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [2]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])
df.dropna(inplace=True)

In [4]:
df.shape

(714, 3)

In [5]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [7]:
X = df.drop('Survived',axis=1)
y = df['Survived']
X.head()

,Age,Fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500


In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=0)

In [9]:
clf = DecisionTreeClassifier()

In [10]:
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
accuracy_score(y_test,y_pred)


0.6033519553072626

In [11]:
np.mean(cross_val_score(clf,X,y,cv=5,scoring='accuracy'))

0.6190584063823501

In [14]:
kbin_age = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='quantile')

In [27]:
trf = ColumnTransformer([
    ('kbin_age',kbin_age,[0]),
    ('kbin_fare',kbin_fare,[1])
])

In [28]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

C:\Users\lalan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lalan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warn

In [29]:
trf.named_transformers_['kbin_age'].bin_edges_

array([array([ 0.67, 18.  , 25.  , 31.  , 42.  , 74.  ])], dtype=object)

In [30]:
trf.named_transformers_['kbin_fare'].bin_edges_

array([array([  0.     ,   7.8958 ,  12.425  ,  25.9292 ,  42.64336, 512.3292 ])],
      dtype=object)

In [20]:
output = pd.DataFrame({
    'Age': X_train['Age'],
    'Fare': X_train['Fare'],
    'Age_bin': X_train_trf[:,0],
    'Fare_bin': X_train_trf[:,1]
    
}
)

In [31]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['kbin_age'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['kbin_fare'].bin_edges_[0].tolist())

In [32]:
output.head()

,Age,Fare,Age_bin,Fare_bin,age_labels,fare_labels
535,7.0,26.2500,0.0,3.0,"(0.67, 18.0]","(25.929, 42.643]"
129,45.0,6.9750,4.0,0.0,"(42.0, 74.0]","(0.0, 7.896]"
491,21.0,7.2500,1.0,0.0,"(18.0, 25.0]","(0.0, 7.896]"
703,25.0,7.7417,2.0,0.0,"(18.0, 25.0]","(0.0, 7.896]"
313,28.0,7.8958,2.0,1.0,"(25.0, 31.0]","(0.0, 7.896]"


In [33]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)
accuracy_score(y_test,y_pred2)


0.6927374301675978

In [34]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(clf,X_trf,y,cv=5,scoring='accuracy'))

C:\Users\lalan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lalan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warn

0.6569486851176991